Code based on: https://www.kaggle.com/code/dmitrykonovalov/fastai-lesson4-getting-started-2024

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

train_df

In [ ]:
test_df

In [ ]:
# Target = 1: tweet describes a real disaster
# Target = 0: tweet does not describe a real disaster

In [ ]:
train_df.describe(include="object")

In [ ]:
train_data = train_df.drop(['id', 'target'], axis=1)
train_data

In [ ]:
train_data_dict = train_data.to_dict(orient="records")
# train_data_dict[:5]

train_data_text = [str(entry) for entry in train_data_dict]
train_data_text[:5]

In [ ]:
test_data = test_df.drop(['id'], axis=1)
test_data

In [ ]:
test_data_dict = test_data.to_dict(orient="records")
# test_data_dict[:5]

test_data_text = [str(entry) for entry in test_data_dict]
test_data_text[:5]

In [ ]:
train_df['input'] = train_data_text
train_df.input.head

In [ ]:
test_df['input'] = test_data_text
test_df.input.head

In [ ]:
from datasets import Dataset, DatasetDict

train_ds = Dataset.from_pandas(train_df)
train_ds

In [ ]:
test_ds = Dataset.from_pandas(test_df)
test_ds

In [ ]:
model_nm = "microsoft/deberta-v3-small"

In [ ]:
from transformers import AutoModelForSequenceClassification,AutoTokenizer
toke = AutoTokenizer.from_pretrained(model_nm)

In [ ]:
def toke_func(x): return toke(x["input"])

In [ ]:
train_toke_ds = train_ds.map(toke_func, batched=True)

In [ ]:
test_toke_ds = test_ds.map(toke_func, batched=True)

In [ ]:
train_toke_ds = train_toke_ds.rename_columns({'target':'labels'})

In [ ]:
dds = train_toke_ds.train_test_split(0.05, seed=42)
dds

In [ ]:
from transformers import TrainingArguments,Trainer

In [ ]:
bs = 64
epochs = 1
lr = 8e-5

In [ ]:
args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, 
                         lr_scheduler_type='cosine', 
                         fp16=True,
    eval_strategy="epoch", 
                         per_device_train_batch_size=bs, 
                         per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, 
                         weight_decay=0.01, 
                         report_to='none')

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
def my_metric(eval_pred): 
    return {'F1': f1_score(*eval_pred, zero_division=0)}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_nm, num_labels=2)

trainer = Trainer(
    model, args, 
    train_dataset=dds['train'], 
    eval_dataset=dds['test'],
                  tokenizer=toke,
)

In [ ]:
trainer.train();

In [ ]:
preds = trainer.predict(test_toke_ds)
preds

In [ ]:
preds.predictions

In [ ]:
class_preds = np.argmax(preds.predictions, axis=-1)
class_preds

In [ ]:
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
sub

In [ ]:
sub['target'] = class_preds
sub

In [ ]:
name = os.path.basename(model_nm)
sub.to_csv(f'submission_{name}_ep{epochs}.csv', index=False)  
sub.to_csv(f'submission.csv', index=False)